In [ ]:
from hdfs import InsecureClient

In [2]:
!hostname -I

172.17.0.2 172.20.0.2 


In [5]:
# 경로내 파일 읽어오기
user = 'hadoop'
host = 'http://namenode:9870'
hdfs_path = '/dataset'
hdfs = InsecureClient(host, user)

show = hdfs.list(hdfs_path)
for s in show:
    print(s)

부산광역시방범용CCTV정보202512.csv
주민등록인구및세대현황202512.csv
행정구역시도성연령별취업자202606.csv


In [ ]:
# 파일읽기
file_name = hdfs_path + "/부산광역시방범용CCTV정보202512.csv"
with hdfs.read(file_name, encoding='cp949') as reader:
    text = reader.read()
print(text[:100])

In [ ]:
# 워드카운트
from collections import Counter

word = text[:5000].strip().split(',')
word_counts = Counter(word)
for word, count in word_counts.items():
    print(word, count)

In [11]:
# 파일업로드
local_file = './dataset/청소년정신건강20260604.csv'
hdfs_file = '/dataset/hdfs청소년정신건강20260604.csv'
hdfs.upload(hdfs_file, local_file, overwrite = True)
print("업로드완료")

업로드완료


In [16]:
# 파일확인
hdfs_file = '/dataset/hdfs청소년정신건강20260604.csv'
if hdfs.status(hdfs_file, strict=False) == None:
    print(f"{hdfs_file[9:]}의 파일이 없습니다.")
else:
    print(hdfs.status(hdfs_file, strict=False))

hdfs청소년정신건강20260604.csv의 파일이 없습니다.


In [18]:
# 파일삭제
hdfs_file = '/dataset/hdfs청소년정신건강20260604.csv'
if hdfs.status(hdfs_file, strict=False) == None:
    print(f"{hdfs_file[9:]}의 파일이 없습니다.")
else:
    hdfs.delete(hdfs_file)
    print("삭제되었습니다.")

hdfs청소년정신건강20260604.csv의 파일이 없습니다.


# HDFS CRUD 전체 코드 정리

| 작업 | 메서드 | 설명 |
|------|--------|------|
| 연결 | `InsecureClient(host, user)` | HDFS 클라이언트 생성 |
| 목록 | `hdfs.list(path)` | 디렉터리 내 파일 목록 조회 |
| 읽기 | `hdfs.read(file, encoding)` | 파일 내용 읽기 |
| 쓰기/업로드 | `hdfs.upload(hdfs_path, local_path, overwrite)` | 로컬 파일을 HDFS에 업로드 |
| 존재확인 | `hdfs.status(path, strict=False)` | 파일 상태 확인 (없으면 None) |
| 삭제 | `hdfs.delete(path)` | 파일 또는 디렉터리 삭제 |

In [ ]:
## 전체 코드 정리 (HDFS CRUD 요약)
from hdfs import InsecureClient

# 1. HDFS 연결
user = 'hadoop'
host = 'http://namenode:9870'
hdfs_path = '/dataset'
hdfs = InsecureClient(host, user)

# 2. 목록 조회 (Read - List)
print("=== 파일 목록 ===")
files = hdfs.list(hdfs_path)
for f in files:
    print(f)

# 3. 파일 읽기 (Read)
print("\n=== 파일 읽기 ===")
file_name = hdfs_path + "/부산광역시방범용CCTV정보202512.csv"
with hdfs.read(file_name, encoding='cp949') as reader:
    text = reader.read()
print(text[:100])

# 4. 파일 업로드 (Create / Update)
print("\n=== 파일 업로드 ===")
local_file = './dataset/청소년정신건강20260604.csv'
hdfs_file = '/dataset/hdfs청소년정신건강20260604.csv'
hdfs.upload(hdfs_file, local_file, overwrite=True)
print("업로드 완료")

# 5. 파일 존재 확인 (Read - Status)
print("\n=== 파일 존재 확인 ===")
status = hdfs.status(hdfs_file, strict=False)
if status is None:
    print(f"{hdfs_file}이 존재하지 않습니다.")
else:
    print(f"파일 정보: {status}")

# 6. 파일 삭제 (Delete)
print("\n=== 파일 삭제 ===")
if hdfs.status(hdfs_file, strict=False) is not None:
    hdfs.delete(hdfs_file)
    print("삭제 완료")
else:
    print("삭제할 파일이 없습니다.")

---
# 개선 연습 코드

아래 코드는 HDFS CRUD를 클래스로 묶어 재사용성을 높이고,  
디렉터리 생성·파일 이동·파일 크기 확인 등 추가 기능을 연습합니다.

In [ ]:
from hdfs import InsecureClient

class HDFSManager:
    """HDFS CRUD 작업을 하나로 묶은 헬퍼 클래스"""

    def __init__(self, host='http://namenode:9870', user='hadoop'):
        self.client = InsecureClient(host, user)
        print(f"HDFS 연결 완료: {host} (user={user})")

    # ── Read ───────────────────────────────────────────────
    def list_files(self, path):
        """디렉터리 파일 목록 반환"""
        files = self.client.list(path, status=True)   # status=True → (이름, 상태) 튜플
        for name, info in files:
            size_kb = info['length'] / 1024
            print(f"  {name:40s}  {size_kb:8.1f} KB  ({info['type']})")
        return [name for name, _ in files]

    def read_file(self, hdfs_path, encoding='utf-8', preview=200):
        """파일 앞부분 미리보기"""
        with self.client.read(hdfs_path, encoding=encoding) as reader:
            content = reader.read()
        print(content[:preview])
        return content

    def file_info(self, hdfs_path):
        """파일 상태 정보 출력"""
        info = self.client.status(hdfs_path, strict=False)
        if info is None:
            print(f"[없음] {hdfs_path}")
        else:
            print(f"[있음] 크기={info['length']} bytes, 블록크기={info['blockSize']}, 수정시각={info['modificationTime']}")
        return info

    # ── Create / Update ────────────────────────────────────
    def make_dir(self, hdfs_path):
        """HDFS 디렉터리 생성"""
        self.client.makedirs(hdfs_path)
        print(f"디렉터리 생성: {hdfs_path}")

    def upload(self, local_path, hdfs_path, overwrite=True):
        """로컬 → HDFS 업로드"""
        self.client.upload(hdfs_path, local_path, overwrite=overwrite)
        print(f"업로드 완료: {local_path} → {hdfs_path}")

    def write_text(self, hdfs_path, content, encoding='utf-8', overwrite=True):
        """문자열을 직접 HDFS 파일로 저장"""
        with self.client.write(hdfs_path, encoding=encoding, overwrite=overwrite) as writer:
            writer.write(content)
        print(f"쓰기 완료: {hdfs_path}")

    # ── Delete ─────────────────────────────────────────────
    def delete(self, hdfs_path, recursive=False):
        """파일 또는 디렉터리 삭제"""
        if self.client.status(hdfs_path, strict=False) is None:
            print(f"[건너뜀] 파일 없음: {hdfs_path}")
        else:
            self.client.delete(hdfs_path, recursive=recursive)
            print(f"삭제 완료: {hdfs_path}")

    # ── Rename / Move ──────────────────────────────────────
    def rename(self, src, dst):
        """HDFS 파일 이름 변경 또는 이동"""
        self.client.rename(src, dst)
        print(f"이동/변경 완료: {src} → {dst}")


# ── 사용 예시 ──────────────────────────────────────────────
mgr = HDFSManager()

print("\n[1] 파일 목록 (크기 포함)")
mgr.list_files('/dataset')

print("\n[2] 파일 정보 확인")
mgr.file_info('/dataset/부산광역시방범용CCTV정보202512.csv')

print("\n[3] 텍스트 직접 쓰기 → 확인 → 삭제")
test_path = '/dataset/test_write.txt'
mgr.write_text(test_path, "HDFS 쓰기 테스트\n라인2\n라인3\n")
mgr.file_info(test_path)
mgr.read_file(test_path, encoding='utf-8')
mgr.delete(test_path)

print("\n[4] 디렉터리 생성 → 파일 업로드 → 이름 변경 → 삭제")
mgr.make_dir('/dataset/practice')
local_csv = './dataset/청소년정신건강20260604.csv'
mgr.upload(local_csv, '/dataset/practice/원본.csv')
mgr.rename('/dataset/practice/원본.csv', '/dataset/practice/변경.csv')
mgr.delete('/dataset/practice/변경.csv')
mgr.delete('/dataset/practice', recursive=True)